#Set up

In [2]:
import re
from typing import Optional
import pandas as pd
import plotly.graph_objects as go

#Função do gráfico

In [8]:
def formatar_numero(valor) -> str:
    """
    Formata números inteiros com separador de milhar em padrão brasileiro.
    """
    return f"{int(round(valor)):,}".replace(",", ".")

def formatar_percentual(valor: float) -> str:
    """
    Formata percentuais com uma casa decimal em padrão brasileiro.
    """
    return f"{valor:.1%}".replace(".", ",")

def extrair_idade_inicial(texto_faixa: str) -> Optional[int]:
    """
    Extrai a idade inicial de uma faixa etária textual.

    Exemplos:
    - "0 a 4 anos" -> 0
    - "15 a 19 anos" -> 15
    - "100 anos ou mais" -> 100
    """
    if pd.isna(texto_faixa):
        return None

    texto_faixa = str(texto_faixa).strip().lower()
    match = re.search(r"\d+", texto_faixa)

    if match:
        return int(match.group())

def classificar_grupo_etario(idade_inicial: Optional[int]) -> Optional[str]:
    """
    Classifica a faixa etária em um macrogrupo.
    """
    if idade_inicial is None:
        return None

    if idade_inicial <= 14:
        return "Jovem"
    if idade_inicial <= 59:
        return "Adulto"
    return "Idoso"

def pertence_populacao_economicamente_ativa(idade_inicial: Optional[int]) -> bool:
    """
    Considera população economicamente ativa como 15 a 64 anos.
    """
    if idade_inicial is None:
        return False
    return 15 <= idade_inicial <= 64


def pertence_idade_fertil_feminina(
    idade_inicial: Optional[int],
    sexo: str,
    sexo_feminino: str,
) -> bool:
    """
    Considera idade fértil feminina como mulheres de 15 a 49 anos.
    """
    if idade_inicial is None:
        return False
    return sexo == sexo_feminino and 15 <= idade_inicial <= 49

In [9]:
def preparar_dados_piramide_etaria(
    df_dados: pd.DataFrame,
    coluna_faixa_etaria: str,
    coluna_sexo: str,
    coluna_valor: str,
    sexo_masculino: str = "Masculino",
    sexo_feminino: str = "Feminino",
    sexo_nao_informado: str = "Não informado",
    ordem_faixas_etarias: Optional[list] = None,
) -> dict:
    """
    Prepara a base da pirâmide etária e todos os resumos auxiliares.

    A função:
    1. valida colunas
    2. trata a coluna numérica
    3. enriquece com grupo etário, PEA e idade fértil
    4. calcula percentuais
    5. monta os rótulos que serão usados no eixo Y
    """
    colunas_obrigatorias = [coluna_faixa_etaria, coluna_sexo, coluna_valor]
    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df_dados.columns]

    if colunas_ausentes:
        raise ValueError(f"Colunas ausentes no DataFrame: {colunas_ausentes}")

    base = df_dados.copy()
    base[coluna_valor] = pd.to_numeric(base[coluna_valor], errors="coerce")
    base = base.dropna(subset=[coluna_valor]).copy()

    if base.empty:
        raise ValueError("Não há dados válidos após a conversão da coluna de valor.")

    if ordem_faixas_etarias is not None:
        base[coluna_faixa_etaria] = pd.Categorical(
            base[coluna_faixa_etaria],
            categories=ordem_faixas_etarias,
            ordered=True,
        )
        base = base.sort_values(coluna_faixa_etaria).copy()
        ordem_final_faixas = [str(x) for x in ordem_faixas_etarias]
    else:
        ordem_final_faixas = list(base[coluna_faixa_etaria].dropna().astype(str).unique())

    base["faixa_str"] = base[coluna_faixa_etaria].astype(str)
    base["idade_inicial_faixa"] = base["faixa_str"].apply(extrair_idade_inicial)
    base["grupo_etario"] = base["idade_inicial_faixa"].apply(classificar_grupo_etario)
    base["eh_pea"] = base["idade_inicial_faixa"].apply(pertence_populacao_economicamente_ativa)
    base["eh_idade_fertil"] = base.apply(
        lambda linha: pertence_idade_fertil_feminina(
            idade_inicial=linha["idade_inicial_faixa"],
            sexo=linha[coluna_sexo],
            sexo_feminino=sexo_feminino,
        ),
        axis=1,
    )

    total_geral = base[coluna_valor].sum()

    totais_por_faixa = (
        base.groupby("faixa_str", observed=False)[coluna_valor]
        .sum()
        .rename("total_faixa")
        .reset_index()
    )

    totais_por_sexo = (
        base.groupby(coluna_sexo, observed=False)[coluna_valor]
        .sum()
        .rename("total_sexo")
        .reset_index()
    )

    base = base.merge(totais_por_faixa, on="faixa_str", how="left")
    base = base.merge(totais_por_sexo, on=coluna_sexo, how="left")

    base["pct_total"] = base[coluna_valor] / total_geral
    base["pct_sexo"] = base[coluna_valor] / base["total_sexo"]
    base["pct_faixa"] = base[coluna_valor] / base["total_faixa"]

    base_masculino = base.loc[base[coluna_sexo] == sexo_masculino].copy()
    base_feminino = base.loc[base[coluna_sexo] == sexo_feminino].copy()
    base_nao_informado = base.loc[base[coluna_sexo] == sexo_nao_informado].copy()

    base_masculino["valor_plotado"] = -base_masculino[coluna_valor]
    base_feminino["valor_plotado"] = base_feminino[coluna_valor]
    base_nao_informado["valor_plotado"] = base_nao_informado[coluna_valor]

    maior_valor_observado = max(
        base_masculino[coluna_valor].max() if not base_masculino.empty else 0,
        base_feminino[coluna_valor].max() if not base_feminino.empty else 0,
        base_nao_informado[coluna_valor].max() if not base_nao_informado.empty else 0,
    )

    resumo_faixas = (
        base.groupby("faixa_str", observed=False)[coluna_valor]
        .sum()
        .reset_index()
        .rename(columns={coluna_valor: "valor_absoluto"})
    )
    resumo_faixas["percentual"] = resumo_faixas["valor_absoluto"] / total_geral

    if ordem_faixas_etarias is not None:
        resumo_faixas["faixa_str"] = pd.Categorical(
            resumo_faixas["faixa_str"],
            categories=ordem_faixas_etarias,
            ordered=True,
        )
        resumo_faixas = resumo_faixas.sort_values("faixa_str").copy()
        resumo_faixas["faixa_str"] = resumo_faixas["faixa_str"].astype(str)

    resumo_faixas["rotulo_eixo_y"] = resumo_faixas.apply(
        lambda linha: (
            f"{linha['faixa_str']}   {formatar_numero(linha['valor_absoluto'])} | "
            f"{formatar_percentual(linha['percentual'])}"
        ),
        axis=1,
    )

    mapa_rotulos_eixo = dict(zip(resumo_faixas["faixa_str"], resumo_faixas["rotulo_eixo_y"]))

    resumo_grupos = (
        base.groupby("grupo_etario", observed=False)[coluna_valor]
        .sum()
        .reset_index()
        .rename(columns={coluna_valor: "valor_absoluto"})
    )
    resumo_grupos["percentual"] = resumo_grupos["valor_absoluto"] / total_geral

    valor_pea = base.loc[base["eh_pea"], coluna_valor].sum()
    pct_pea = valor_pea / total_geral if total_geral > 0 else 0

    valor_idade_fertil = base.loc[base["eh_idade_fertil"], coluna_valor].sum()
    pct_idade_fertil = valor_idade_fertil / total_geral if total_geral > 0 else 0

    total_feminino = base_feminino[coluna_valor].sum() if not base_feminino.empty else 0
    pct_idade_fertil_no_feminino = valor_idade_fertil / total_feminino if total_feminino > 0 else 0

    return {
        "base_tratada": base,
        "base_masculino": base_masculino,
        "base_feminino": base_feminino,
        "base_nao_informado": base_nao_informado,
        "total_geral": total_geral,
        "maior_valor_observado": maior_valor_observado,
        "ordem_final_faixas": ordem_final_faixas,
        "mapa_rotulos_eixo": mapa_rotulos_eixo,
        "resumo_grupos": resumo_grupos,
        "valor_pea": valor_pea,
        "pct_pea": pct_pea,
        "valor_idade_fertil": valor_idade_fertil,
        "pct_idade_fertil": pct_idade_fertil,
        "pct_idade_fertil_no_feminino": pct_idade_fertil_no_feminino,
    }

In [58]:
def construir_figura_piramide_etaria(
    dados_preparados: dict,
    coluna_faixa_etaria: str,
    coluna_valor: str,
    titulo: str = "Pirâmide etária com resumos laterais",
    titulo_eixo_y: str = "Faixa etária",
    sexo_masculino: str = "Masculino",
    sexo_feminino: str = "Feminino",
    sexo_nao_informado: str = "Não informado",
    incluir_nao_informado: bool = True,
    mostrar_absoluto_na_barra: bool = True,
    mostrar_percentual_total_na_barra: bool = True,
    mostrar_percentual_sexo_externo: bool = True,
    mostrar_percentual_faixa_externo: bool = True,
    largura: int = 1300,
    altura: int = 1000,
    cor_masculino: str = "#5B84D6",
    cor_feminino: str = "#D97190",
    cor_nao_informado: str = "#9E9E9E",
    tamanho_fonte_texto_interno: int = 12,
    tamanho_fonte_texto_externo: int = 10,
    tamanho_fonte_titulo: int = 24,
    fator_folga_eixo_x: float = 1.50,
    fator_deslocamento_texto_externo: float = 0.035,
    proporcao_minima_rotulo_interno: float = 0.20,
    proporcao_minima_sem_rotulo: float = 0.0,
) -> go.Figure:
    """
    Constrói a figura da pirâmide etária.

    Regras desta versão:
    - o eixo Y carrega faixa etária, absoluto e percentual
    - rótulos pequenos podem sair da barra
    - rótulos muito pequenos são ocultados para evitar poluição visual,
      já que a informação já aparece no eixo Y
    - quando o rótulo principal sai da barra, o percentual lateral anda ainda mais
      para fora e pode receber deslocamento vertical
    - caixas de apoio ficam no topo direito, fora da área útil do gráfico
    """
    base_masculino = dados_preparados["base_masculino"]
    base_feminino = dados_preparados["base_feminino"]
    base_nao_informado = dados_preparados["base_nao_informado"]
    total_geral = dados_preparados["total_geral"]
    maior_valor_observado = dados_preparados["maior_valor_observado"]
    ordem_final_faixas = dados_preparados["ordem_final_faixas"]
    mapa_rotulos_eixo = dados_preparados["mapa_rotulos_eixo"]
    resumo_grupos = dados_preparados["resumo_grupos"]
    valor_pea = dados_preparados["valor_pea"]
    pct_pea = dados_preparados["pct_pea"]
    valor_idade_fertil = dados_preparados["valor_idade_fertil"]
    pct_idade_fertil = dados_preparados["pct_idade_fertil"]
    pct_idade_fertil_no_feminino = dados_preparados["pct_idade_fertil_no_feminino"]

    def montar_rotulo_barra(linha: pd.Series) -> str:
        partes = []
        if mostrar_absoluto_na_barra:
            partes.append(formatar_numero(linha[coluna_valor]))
        if mostrar_percentual_total_na_barra:
            partes.append(formatar_percentual(linha["pct_total"]))
        return " | ".join(partes)

    def montar_texto_externo(linha: pd.Series) -> str:
        partes = []
        if mostrar_percentual_sexo_externo:
            partes.append(f"{formatar_percentual(linha['pct_sexo'])} do sexo")
        if mostrar_percentual_faixa_externo:
            partes.append(f"{formatar_percentual(linha['pct_faixa'])} da faixa")
        return "<br>".join(partes)

    def adicionar_barras(figura, df_local, nome, cor, opacidade=1.0):
        if df_local.empty:
            return
        figura.add_trace(
            go.Bar(
                y=df_local["faixa_str"],
                x=df_local["valor_plotado"],
                name=nome,
                orientation="h",
                marker=dict(color=cor),
                text=None,
                hovertemplate=(
                    "<b>%{y}</b><br>"
                    "Valor: %{customdata[0]}<br>"
                    "Percentual do total: %{customdata[1]}<br>"
                    "Percentual do sexo: %{customdata[2]}<br>"
                    "Percentual da faixa: %{customdata[3]}<extra></extra>"
                ),
                customdata=df_local[
                    [coluna_valor, "pct_total", "pct_sexo", "pct_faixa"]
                ].assign(
                    **{
                        coluna_valor: lambda x: x[coluna_valor].apply(formatar_numero),
                        "pct_total": lambda x: x["pct_total"].apply(formatar_percentual),
                        "pct_sexo": lambda x: x["pct_sexo"].apply(formatar_percentual),
                        "pct_faixa": lambda x: x["pct_faixa"].apply(formatar_percentual),
                    }
                ).values,
                opacity=opacidade,
            )
        )

    def adicionar_rotulos_barras(figura, df_local, lado, cor_interno, cor_externo, limite_eixo_x):
        """
        Adiciona o rótulo principal de cada barra.

        Estratégia final:
        - barras grandes: rótulo dentro
        - barras médias: rótulo fora, na mesma linha
        - barras muito pequenas: rótulo omitido

        Importante:
        - nesta versão não usamos yshift para o rótulo principal
        - quando o rótulo sai da barra, apenas registramos a largura estimada
          para afastar mais o texto lateral de percentuais
        """
        mapa_posicao = {}
        if df_local.empty:
            return mapa_posicao

        for _, linha in df_local.iterrows():
            texto = montar_rotulo_barra(linha)
            if not texto:
                continue

            valor_absoluto = abs(linha["valor_plotado"])
            proporcao = valor_absoluto / limite_eixo_x if limite_eixo_x > 0 else 0
            largura_estimada = max(len(texto) * limite_eixo_x * 0.010, limite_eixo_x * 0.070)

            chave = str(linha["faixa_str"])

            if proporcao < proporcao_minima_sem_rotulo:
                mapa_posicao[chave] = {
                    "posicao": "omitido",
                    "largura_estimada": 0.0,
                    "x_rotulo": None,
                    "yshift": 0,
                }
                continue

            if proporcao >= proporcao_minima_rotulo_interno:
                posicao = "interno"
                x = linha["valor_plotado"] / 2
                xanchor = "center"
                cor = cor_interno
            else:
                posicao = "externo"
                offset = limite_eixo_x * 0.018
                if lado == "esquerda":
                    x = linha["valor_plotado"] - offset
                    xanchor = "right"
                else:
                    x = linha["valor_plotado"] + offset
                    xanchor = "left"

                # Para rótulos externos, estimamos a largura do texto em unidades do eixo X.
                # Essa medida será usada depois para empurrar o texto lateral de percentuais
                # para além do rótulo principal, evitando sobreposição.
                largura_estimada = max(len(texto) * limite_eixo_x * 0.0135, limite_eixo_x * 0.125)
                cor = cor_externo

            figura.add_annotation(
                x=x,
                y=linha["faixa_str"],
                text=texto,
                showarrow=False,
                xanchor=xanchor,
                yanchor="middle",
                align="center",
                yshift=0,
                font=dict(size=tamanho_fonte_texto_interno, color=cor),
            )

            mapa_posicao[chave] = {
                "posicao": posicao,
                "largura_estimada": largura_estimada,
                "x_rotulo": x,
                "yshift": 0,
            }

        return mapa_posicao

    def adicionar_anotacoes_externas(figura, df_local, lado, cor_texto, limite_eixo_x, mapa_rotulos_principais):
        """
        Adiciona percentuais externos.

        Regra desta versão:
        - não desloca verticalmente
        - quando o rótulo principal ficou fora da barra, o percentual lateral é
          afastado horizontalmente para não colidir com ele
        - quando o rótulo principal foi omitido, o percentual lateral ocupa a
          posição padrão, mais próxima da barra
        """
        if df_local.empty:
            return

        for _, linha in df_local.iterrows():
            texto = montar_texto_externo(linha)
            if not texto:
                continue

            chave = str(linha["faixa_str"])
            info_rotulo = mapa_rotulos_principais.get(chave, {})
            posicao_rotulo = info_rotulo.get("posicao", "interno")
            largura_estimada = info_rotulo.get("largura_estimada", 0.0)

            deslocamento_base = limite_eixo_x * fator_deslocamento_texto_externo
            espacamento_extra_rotulo_externo = limite_eixo_x * 0.085

            if lado == "esquerda":
                ancora_x = "right"
                alinhamento = "right"

                if posicao_rotulo == "externo" and info_rotulo.get("x_rotulo") is not None:
                    # Quando o rótulo principal saiu da barra, o texto lateral deve
                    # começar apenas depois do fim desse rótulo, com uma folga adicional.
                    posicao_x = (
                        info_rotulo["x_rotulo"]
                        - largura_estimada
                        - deslocamento_base
                        - espacamento_extra_rotulo_externo
                    )
                else:
                    posicao_x = linha["valor_plotado"] - deslocamento_base

            else:
                ancora_x = "left"
                alinhamento = "left"

                if posicao_rotulo == "externo" and info_rotulo.get("x_rotulo") is not None:
                    posicao_x = (
                        info_rotulo["x_rotulo"]
                        + largura_estimada
                        + deslocamento_base
                        + espacamento_extra_rotulo_externo
                    )
                else:
                    posicao_x = linha["valor_plotado"] + deslocamento_base

            figura.add_annotation(
                x=posicao_x,
                y=linha["faixa_str"],
                text=texto,
                showarrow=False,
                xanchor=ancora_x,
                yanchor="middle",
                align=alinhamento,
                yshift=0,
                font=dict(size=tamanho_fonte_texto_externo, color=cor_texto),
            )

    def obter_valor_grupo(nome_grupo: str):
        linha = resumo_grupos.loc[resumo_grupos["grupo_etario"] == nome_grupo]
        if linha.empty:
            return 0, 0.0
        return linha["valor_absoluto"].iloc[0], linha["percentual"].iloc[0]

    limite_eixo_x = maior_valor_observado * fator_folga_eixo_x if maior_valor_observado > 0 else 1

    figura = go.Figure()
    adicionar_barras(figura, base_masculino, sexo_masculino, cor_masculino)
    adicionar_barras(figura, base_feminino, sexo_feminino, cor_feminino)

    if incluir_nao_informado:
        adicionar_barras(figura, base_nao_informado, sexo_nao_informado, cor_nao_informado, 0.85)

    tickvals = ordem_final_faixas
    ticktext = [mapa_rotulos_eixo.get(faixa, faixa) for faixa in ordem_final_faixas]

    figura.update_layout(
        title=dict(
            text=f"<b>{titulo}</b>",
            x=0.03,
            xanchor="left",
            font=dict(size=tamanho_fonte_titulo),
        ),
        barmode="relative",
        template="plotly_white",
        width=1700,
        height=altura,
        xaxis=dict(
            showgrid=False,
            showticklabels=False,
            title=None,
            zeroline=False,
            visible=False,
            range=[-limite_eixo_x, limite_eixo_x],
        ),
        yaxis=dict(
            title=titulo_eixo_y,
            categoryorder="array",
            categoryarray=ordem_final_faixas,
            tickmode="array",
            tickvals=tickvals,
            ticktext=ticktext,
            tickfont=dict(size=11, color="#1F3558"),
            automargin=True,
        ),
        legend=dict(
            orientation="h",
            y=1.04,
            x=0.50,
            xanchor="center",
        ),
        margin=dict(l=250, r=500, t=100, b=120),
        plot_bgcolor="white",
        paper_bgcolor="white",
    )

    figura.add_vline(x=0, line_width=1.0, line_color="rgba(120,120,120,0.8)")

    mapa_rotulos_masc = adicionar_rotulos_barras(figura, base_masculino, "esquerda", "white", "#2F3B52", limite_eixo_x)
    mapa_rotulos_fem = adicionar_rotulos_barras(figura, base_feminino, "direita", "white", "#8C2F4E", limite_eixo_x)
    mapa_rotulos_nao_inf = {}
    if incluir_nao_informado:
        mapa_rotulos_nao_inf = adicionar_rotulos_barras(figura, base_nao_informado, "direita", "white", "#555555", limite_eixo_x)

    adicionar_anotacoes_externas(figura, base_masculino, "esquerda", "#2F3B52", limite_eixo_x, mapa_rotulos_masc)
    adicionar_anotacoes_externas(figura, base_feminino, "direita", "#6B2D42", limite_eixo_x, mapa_rotulos_fem)
    if incluir_nao_informado:
        adicionar_anotacoes_externas(figura, base_nao_informado, "direita", "#555555", limite_eixo_x, mapa_rotulos_nao_inf)

    total_masculino = base_masculino[coluna_valor].sum() if not base_masculino.empty else 0
    total_feminino = base_feminino[coluna_valor].sum() if not base_feminino.empty else 0

    texto_resumo_superior = (
        f"<b>Feminino:</b> {formatar_numero(total_feminino)} ({formatar_percentual(total_feminino / total_geral)})"
        f" &nbsp;&nbsp;&nbsp;&nbsp; "
        f"<b>Masculino:</b> {formatar_numero(total_masculino)} ({formatar_percentual(total_masculino / total_geral)})"
        f" &nbsp;&nbsp;&nbsp;&nbsp; "
        f"<b>Total:</b> {formatar_numero(total_geral)}"
    )

    figura.add_annotation(
        x=0.20,
        y=1.10,
        xref="paper",
        yref="paper",
        text=texto_resumo_superior,
        showarrow=False,
        align="left",
        xanchor="left",
        font=dict(size=14, color="#333333"),
    )

    valor_jovem, pct_jovem = obter_valor_grupo("Jovem")
    valor_adulto, pct_adulto = obter_valor_grupo("Adulto")
    valor_idoso, pct_idoso = obter_valor_grupo("Idoso")

    texto_caixa_grupos = (
        f"<b>Jovem</b><br>{formatar_numero(valor_jovem)} | {formatar_percentual(pct_jovem)}"
        f"<br><br>"
        f"<b>Adulto</b><br>{formatar_numero(valor_adulto)} | {formatar_percentual(pct_adulto)}"
        f"<br><br>"
        f"<b>Idoso</b><br>{formatar_numero(valor_idoso)} | {formatar_percentual(pct_idoso)}"
    )

    figura.add_annotation(
        x=1.00,
        y=0.92,
        xref="paper",
        yref="paper",
        text=texto_caixa_grupos,
        showarrow=False,
        align="left",
        xanchor="left",
        yanchor="top",
        bordercolor="#BFBFBF",
        borderwidth=1,
        bgcolor="#F7F7F7",
        font=dict(size=12, color="#333333"),
    )

    texto_caixa_especiais = (
        f"<b>População economicamente ativa</b><br>"
        f"{formatar_numero(valor_pea)} | {formatar_percentual(pct_pea)}"
        f"<br><br>"
        f"<b>Idade fértil feminina</b><br>"
        f"{formatar_numero(valor_idade_fertil)} | {formatar_percentual(pct_idade_fertil)}"
        f"<br>Do feminino: {formatar_percentual(pct_idade_fertil_no_feminino)}"
    )

    figura.add_annotation(
        x=1.00,
        y=0.62,
        xref="paper",
        yref="paper",
        text=texto_caixa_especiais,
        showarrow=False,
        align="left",
        xanchor="left",
        yanchor="top",
        bordercolor="#BFBFBF",
        borderwidth=1,
        bgcolor="#F7F7F7",
        font=dict(size=11.5, color="#333333"),
    )

    texto_definicoes = (
        "<b>Definições</b><br>"
        "Jovem: 0 a 14 anos<br>"
        "Adulto: 15 a 59 anos<br>"
        "Idoso: 60 anos ou mais<br>"
        "População economicamente ativa: 15 a 64 anos<br>"
        "Idade fértil feminina: mulheres de 15 a 49 anos"
    )

    figura.add_annotation(
        x=1.00,
        y=0.33,
        xref="paper",
        yref="paper",
        text=texto_definicoes,
        showarrow=False,
        align="left",
        xanchor="left",
        yanchor="top",
        bordercolor="#BFBFBF",
        borderwidth=1,
        bgcolor="#F7F7F7",
        font=dict(size=11, color="#333333"),
    )

    figura.add_annotation(
        x=0.50,
        y=-0.10,
        xref="paper",
        yref="paper",
        text=(
            "No eixo Y: faixa etária, absoluto e percentual do total"
            " | Dentro da barra: absoluto e percentual do total quando houver espaço"
            " | Fora da barra: percentual dentro do sexo e dentro da faixa"
        ),
        showarrow=False,
        align="center",
        font=dict(size=11, color="#555555"),
    )

    return figura


def grafico_piramide_etaria(
    df_dados: pd.DataFrame,
    coluna_faixa_etaria: str,
    coluna_sexo: str,
    coluna_valor: str,
    titulo: str = "Pirâmide etária com resumos laterais",
    titulo_eixo_y: str = "Faixa etária",
    ordem_faixas_etarias: Optional[list] = None,
    sexo_masculino: str = "Masculino",
    sexo_feminino: str = "Feminino",
    sexo_nao_informado: str = "Não informado",
    incluir_nao_informado: bool = True,
) -> go.Figure:
    """
    Função principal de orquestração.
    """
    dados_preparados = preparar_dados_piramide_etaria(
        df_dados=df_dados,
        coluna_faixa_etaria=coluna_faixa_etaria,
        coluna_sexo=coluna_sexo,
        coluna_valor=coluna_valor,
        sexo_masculino=sexo_masculino,
        sexo_feminino=sexo_feminino,
        sexo_nao_informado=sexo_nao_informado,
        ordem_faixas_etarias=ordem_faixas_etarias,
    )

    figura = construir_figura_piramide_etaria(
        dados_preparados=dados_preparados,
        coluna_faixa_etaria=coluna_faixa_etaria,
        coluna_valor=coluna_valor,
        titulo=titulo,
        titulo_eixo_y=titulo_eixo_y,
        sexo_masculino=sexo_masculino,
        sexo_feminino=sexo_feminino,
        sexo_nao_informado=sexo_nao_informado,
        incluir_nao_informado=incluir_nao_informado,
    )

    return figura


# Dados de demostração

In [59]:

# Exemplo de dados para teste

ordem_faixas = [
    "0 a 4 anos",
    "5 a 9 anos",
    "10 a 14 anos",
    "15 a 19 anos",
    "20 a 24 anos",
    "25 a 29 anos",
    "30 a 34 anos",
    "35 a 39 anos",
    "40 a 44 anos",
    "45 a 49 anos",
    "50 a 54 anos",
    "55 a 59 anos",
    "60 a 64 anos",
    "65 a 69 anos",
    "70 a 74 anos",
    "75 a 79 anos",
    "80 a 84 anos",
    "85 a 89 anos",
    "90 a 94 anos",
    "95 a 99 anos",
    "100 anos ou mais",
]

valores_masculino = [
    64000, 69000, 65000, 79000, 88000, 93000, 97000, 92000, 86000, 83000,
    78000, 72000, 60000, 47000, 34000, 21000, 11000, 5000, 1800, 500, 80
]

valores_feminino = [
    61000, 67000, 62000, 82000, 91000, 95000, 98000, 94000, 89000, 86000,
    82000, 76000, 64000, 51000, 39000, 26000, 14000, 7000, 2500, 780, 120
]

registros = []
for faixa, valor_m, valor_f in zip(ordem_faixas, valores_masculino, valores_feminino):
    registros.append({"faixa_etaria": faixa, "genero": "Masculino", "qtd_ativos": valor_m})
    registros.append({"faixa_etaria": faixa, "genero": "Feminino", "qtd_ativos": valor_f})

df_teste_piramide = pd.DataFrame(registros)
df_teste_piramide.head()


,faixa_etaria,genero,qtd_ativos
0,0 a 4 anos,Masculino,64000
1,0 a 4 anos,Feminino,61000
2,5 a 9 anos,Masculino,69000
3,5 a 9 anos,Feminino,67000
4,10 a 14 anos,Masculino,65000


# Plotagem

In [60]:

fig = grafico_piramide_etaria(
    df_dados=df_teste_piramide,
    coluna_faixa_etaria="faixa_etaria",
    coluna_sexo="genero",
    coluna_valor="qtd_ativos",
    titulo="Pirâmide etária",
    titulo_eixo_y="Faixa etária",
    ordem_faixas_etarias=ordem_faixas,
)

fig.show()


In [47]:

# Célula opcional para validar os resumos calculados

dados_preparados = preparar_dados_piramide_etaria(
    df_dados=df_teste_piramide,
    coluna_faixa_etaria="faixa_etaria",
    coluna_sexo="genero",
    coluna_valor="qtd_ativos",
    ordem_faixas_etarias=ordem_faixas,
)

display(dados_preparados["resumo_grupos"])


,grupo_etario,valor_absoluto,percentual
0,Adulto,1561000,0.668872
1,Idoso,384780,0.164874
2,Jovem,388000,0.166254
